In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Disable auto-scroll in notebook output
display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def rational_sampling_freq_simulation(L, M):
    clear_output(wait=True)
    
    # 1. Frequency vector covering [-1.5*pi, 1.5*pi]
    N_fft = 4000
    w = np.linspace(-1.5 * np.pi, 1.5 * np.pi, N_fft)
    
    # Base Signal Spectrum X(e^{j\omega_x}): Triangular spectrum with bandwidth 0.4*pi
    omega_N = 0.4 * np.pi
    
    def base_spectrum(w_axis):
        spec = np.zeros_like(w_axis)
        mask = np.abs(w_axis) <= omega_N
        spec[mask] = 1.0 - np.abs(w_axis[mask]) / omega_N
        return spec

    # Original Spectrum X(e^{j\omega_x})
    x_spec = base_spectrum(w)
    
    # 2. Expander (Interpolation by L): V(e^{j\omega_v}) = L * X(e^{j\omega_v * L})
    # Frequency axis is compressed by L around each replica center (2*pi*k / L)
    v_spec = np.zeros_like(w)
    for k in range(-10, 11):
        replica_center = k * (2.0 * np.pi / L)
        v_spec += L * base_spectrum((w - replica_center) * L)
    
    # 3. Ideal Low-Pass Filter: Cutoff frequency min(pi/L, pi/M)
    omega_c = min(np.pi / L, np.pi / M)
    h_spec = np.where(np.abs(w) <= omega_c, 1.0, 0.0)
    w_spec = v_spec * h_spec
    
    # 4. Decimator (Downsampling by M): Stretches frequency axis by M and scales amplitude by 1/M
    y_spec = np.zeros_like(w)
    for k in range(-5, 6):
        y_spec += (1.0 / M) * np.interp((w - 2.0 * np.pi * k) / M, w, w_spec, left=0, right=0)
    
    # --- Plotting ---
    fig, axes = plt.subplots(3, 1, figsize=(12, 9))
    y_max = max(1.2, L * 1.1)
    
    # Plot 1: Original Spectrum X(e^{j\omega_x})
    axes[0].plot(w / np.pi, x_spec, color='tab:blue', lw=2.5, label=r'Original Spectrum $\mathcal{X}(e^{j\omega_x})$')
    axes[0].fill_between(w / np.pi, 0, x_spec, color='tab:blue', alpha=0.2)
    axes[0].set_title(r'1. Original Spectrum $\mathcal{X}(e^{j\omega_x})$', fontsize=10, fontweight='bold')
    axes[0].set_ylabel('Amplitude', fontsize=9)
    axes[0].set_xlim(-1.5, 1.5)
    axes[0].set_ylim(-0.05, y_max)
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, fontsize=9)
    
    # Plot 2: Expanded Spectrum with Replicas BEFORE and AFTER LPF
    axes[1].plot(w / np.pi, v_spec, color='purple', linestyle='--', lw=1.5, label=r'Expanded Spectrum with Replicas ($L=' + str(L) + '$)')
    axes[1].plot(w / np.pi, w_spec, color='tab:red', lw=2.5, label=r'After Low-Pass Filter')
    axes[1].fill_between(w / np.pi, 0, w_spec, color='tab:red', alpha=0.3)
    axes[1].axvline(x=omega_c / np.pi, color='green', linestyle=':', lw=2, label=r'Filter Cutoff $\min(\pi/L, \pi/M)$')
    axes[1].axvline(x=-omega_c / np.pi, color='green', linestyle=':', lw=2)
    axes[1].set_title(r'2. Expanded Spectrum ($L=' + str(L) + r'$) & Replicas Before/After Filtering', fontsize=10, fontweight='bold')
    axes[1].set_ylabel('Amplitude', fontsize=9)
    axes[1].set_xlim(-1.5, 1.5)
    axes[1].set_ylim(-0.05, y_max)
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, fontsize=9)
    
    # Plot 3: Final Output Spectrum Y(e^{j\omega_y})
    axes[2].plot(w / np.pi, y_spec, color='tab:green', lw=2.5, label=r'Final Spectrum $\mathcal{Y}(e^{j\omega_y})$')
    axes[2].fill_between(w / np.pi, 0, y_spec, color='tab:green', alpha=0.2)
    axes[2].set_title(r'3. Final Output Spectrum ($M=' + str(M) + r'$), Effective Ratio $L/M = ' + f'{L}/{M} = {L/M:.2f}$', fontsize=10, fontweight='bold')
    axes[2].set_xlabel(r'Normalized Frequency ($\omega / \pi$)', fontsize=9)
    axes[2].set_ylabel('Amplitude', fontsize=9)
    axes[2].set_xlim(-1.5, 1.5)
    axes[2].set_ylim(-0.05, y_max)
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True, fontsize=9)
    
    plt.tight_layout()
    plt.show()

# Interactive Sliders for L and M
l_slider = widgets.IntSlider(value=2, min=1, max=5, step=1, description='Interpolation (L):', style={'description_width': 'initial'})
m_slider = widgets.IntSlider(value=2, min=1, max=5, step=1, description='Decimation (M):', style={'description_width': 'initial'})

ui = widgets.VBox([l_slider, m_slider])
display(ui)

out = widgets.interactive_output(rational_sampling_freq_simulation, {'L': l_slider, 'M': m_slider})
display(out)